In [1]:
import os
from d3rlpy.logging import UnifiedFileAdapterFactory
import numpy as np
local = False
if local:
    os.environ["D3RLPY_DATASETS_PATH"] = "/home/julian/programming_data/d3rlpy_data"
    os.environ["MINARI_DATASETS_PATH"] = "/home/julian/programming_data/d3rlpy_data/minari_data"
else:
    # on cluster
    os.environ["D3RLPY_DATASETS_PATH"] = "/gpfs/data/fs72297/jklotz/programming_data/d3rlpy_data"
    os.environ["MINARI_DATASETS_PATH"] = "/gpfs/data/fs72297/jklotz/programming_data/d3rlpy_data/minari_data"
os.makedirs(os.environ["D3RLPY_DATASETS_PATH"], exist_ok=True)
os.makedirs(os.environ["MINARI_DATASETS_PATH"], exist_ok=True)

#os.environ["CUDA_LAUNCH_BLOCKING"]="1"
import d3rlpy

dataset, env = d3rlpy.datasets.get_minari('atari/pong/expert-v0')
seed = 1



dataset_name = "pong"


# settings in original DT paper
# Batch size 512 Pong 128 Breakout, Qbert, Seaquest Context length K 50 Pong 30 Breakout, Qbert, Seaquest Return-to-go conditioning 90 Breakout (≈ 1× max in dataset) 2500 Qbert (≈ 5× max in dataset) 20 Pong (≈ 1× max in dataset) 1450 Seaquest (≈ 5× max in dataset)

if dataset_name == "pong":
    batch_size = 8
    context_size = 50
    target_return = 20
elif dataset_name == "breakout":
    batch_size = 128
    context_size = 30
    target_return = 90
elif dataset_name == "qbert":
    batch_size = 128
    context_size = 30
    target_return = 2500
elif dataset_name == "seaquest":    
    batch_size = 128
    context_size = 30
    target_return = 1450

# fix seed
d3rlpy.seed(seed)
d3rlpy.envs.seed_env(env, seed)

dt = d3rlpy.algos.DiscreteDecisionTransformerConfig(
    batch_size=batch_size,
    context_size=context_size,
    learning_rate=6e-4,
    activation_type="gelu",
    embed_activation_type="tanh",
    encoder_factory=d3rlpy.models.PixelEncoderFactory(
        feature_size=128, exclude_last_activation=True
    ),  # Nature DQN
    num_heads=2,#8
    num_layers=1,#4
    attn_dropout=0.1,
    embed_dropout=0.1,
    optim_factory=d3rlpy.optimizers.GPTAdamWFactory(
        betas=(0.9, 0.95),
        weight_decay=0.1,
        clip_grad_norm=1.0,
    ),
    warmup_tokens=512 * 20,
    final_tokens=2 * 500000 * context_size * 3,
    observation_scaler=d3rlpy.preprocessing.PixelObservationScaler(),
    max_timestep=2000, # TODO
    position_encoding_type=d3rlpy.PositionEncodingType.GLOBAL,
    compile_graph=False,
).create(device="cpu")

dt.fit(
    dataset,
    n_steps=100,#100000,
    n_steps_per_epoch=10,#1000,
    save_interval=100,
    eval_env=env,
    eval_target_return=target_return,
    experiment_name=f"Discrete_DT_Cartpole",
    n_trials=50,
    eval_gaps=1,
    eval_action_sampler=d3rlpy.algos.SoftmaxTransformerActionSampler(
            temperature=1.0,
        ),
    logger_adapter=UnifiedFileAdapterFactory(),
)

/gpfs/data/fs72297/jklotz/.conda/envs/d3rlpy_dev_requirements_py310/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]


2025-07-31 10:01.09 [info     ] Signatures have been automatically determined. action_signature=Signature(dtype=[dtype('int64')], shape=[(1,)]) observation_signature=Signature(dtype=[dtype('uint8')], shape=[(3, 210, 160)]) reward_signature=Signature(dtype=[dtype('float64')], shape=[(1,)])
2025-07-31 10:01.09 [info     ] Action-space has been automatically determined. action_space=<ActionSpace.DISCRETE: 2>
2025-07-31 10:01.09 [info     ] Action size has been automatically determined. action_size=6
2025-07-31 10:01.09 [info     ] dataset info                   dataset_info=DatasetInfo(observation_signature=Signature(dtype=[dtype('uint8')], shape=[(3, 210, 160)]), action_signature=Signature(dtype=[dtype('int64')], shape=[(1,)]), reward_signature=Signature(dtype=[dtype('float64')], shape=[(1,)]), action_space=<ActionSpace.DISCRETE: 2>, action_size=6)
2025-07-31 10:01.09 [debug    ] Building models...            
2025-07-31 10:01.10 [debug    ] Models have been built.       
2025-07-31 10:0

Epoch 1/10: 100%|██████████| 10/10 [00:04<00:00,  2.01it/s, loss=1.81, learning_rate=2.34e-5]


2025-07-31 10:01.36 [info     ] New best score                 epoch=1 score=-21.0
2025-07-31 10:01.36 [info     ] Saving model 'd3rlpy_logs/Discrete_DT_Cartpole_20250731100110/model_epoch_1.d3' epoch=1
2025-07-31 10:01.36 [info     ] Model parameters are saved to d3rlpy_logs/Discrete_DT_Cartpole_20250731100110/model_epoch_1.d3
2025-07-31 10:01.36 [info     ] Discrete_DT_Cartpole_20250731100110: epoch=1 step=10 epoch=1 metrics={'time_sample_batch': 0.023603940010070802, 'time_algorithm_update': 0.47244038581848147, 'loss': 1.7811839580535889, 'learning_rate': 0.000126744140625, 'time_step': 0.49616641998291017, 'eval_episode_mean_reward': -21.0, 'eval_episode_median_reward': -21.0, 'eval_episode_std_reward': 0.0, 'eval_episode_min_reward': -21.0, 'eval_episode_max_reward': -21.0, 'eval_episode_count': 1.0} step=10


Epoch 2/10: 100%|██████████| 10/10 [00:04<00:00,  2.08it/s, loss=1.8, learning_rate=0.000252]


...

2025-07-31 12:10.28 [info     ] Discrete_DT_Cartpole_20250731100110: epoch=8 step=80 epoch=8 metrics={'time_sample_batch': 0.02207207679748535, 'time_algorithm_update': 0.4390546798706055, 'loss': 1.7061801433563233, 'learning_rate': 0.0005999999747754388, 'time_step': 0.4612403154373169, 'eval_episode_mean_reward': -20.6, 'eval_episode_median_reward': -21.0, 'eval_episode_std_reward': 0.5291502622129182, 'eval_episode_min_reward': -21.0, 'eval_episode_max_reward': -19.0, 'eval_episode_count': 50.0} step=80


Epoch 9/10: 100%|██████████| 10/10 [00:04<00:00,  2.02it/s, loss=1.71, learning_rate=0.0006]


2025-07-31 12:28.55 [info     ] Discrete_DT_Cartpole_20250731100110: epoch=9 step=90 epoch=9 metrics={'time_sample_batch': 0.025771117210388182, 'time_algorithm_update': 0.469241738319397, 'loss': 1.6875514745712281, 'learning_rate': 0.0005999999636287791, 'time_step': 0.4951273679733276, 'eval_episode_mean_reward': -20.44, 'eval_episode_median_reward': -21.0, 'eval_episode_std_reward': 0.7525955088890712, 'eval_episode_min_reward': -21.0, 'eval_episode_max_reward': -18.0, 'eval_episode_count': 50.0} step=90


Epoch 10/10: 100%|██████████| 10/10 [00:04<00:00,  2.05it/s, loss=1.71, learning_rate=0.0006]


2025-07-31 12:48.05 [info     ] Discrete_DT_Cartpole_20250731100110: epoch=10 step=100 epoch=10 metrics={'time_sample_batch': 0.02502458095550537, 'time_algorithm_update': 0.4624243497848511, 'loss': 1.6972237467765807, 'learning_rate': 0.0005999999504493323, 'time_step': 0.4875725030899048, 'eval_episode_mean_reward': -20.4, 'eval_episode_median_reward': -20.5, 'eval_episode_std_reward': 0.6633249580710799, 'eval_episode_min_reward': -21.0, 'eval_episode_max_reward': -19.0, 'eval_episode_count': 50.0} step=100


In [5]:
print("Max timestep in dataset:", max([len(ep.actions) for ep in dataset.episodes]))

Max timestep in dataset: 1962


In [5]:
for ep in dataset.episodes:
    print(ep.observations.shape)
    break

(1864, 3, 210, 160)


In [6]:
env.observation_space

Box(0, 255, (210, 160, 3), uint8)

In [9]:
type(env)

gymnasium.wrappers.common.TimeLimit

In [1]:
import os
from d3rlpy.logging import UnifiedFileAdapterFactory
import numpy as np
local = False
if local:
    os.environ["D3RLPY_DATASETS_PATH"] = "/home/julian/programming_data/d3rlpy_data"
    os.environ["MINARI_DATASETS_PATH"] = "/home/julian/programming_data/d3rlpy_data/minari_data"
else:
    # on cluster
    os.environ["D3RLPY_DATASETS_PATH"] = "/gpfs/data/fs72297/jklotz/programming_data/d3rlpy_data"
    os.environ["MINARI_DATASETS_PATH"] = "/gpfs/data/fs72297/jklotz/programming_data/d3rlpy_data/minari_data"
os.makedirs(os.environ["D3RLPY_DATASETS_PATH"], exist_ok=True)
os.makedirs(os.environ["MINARI_DATASETS_PATH"], exist_ok=True)

#os.environ["CUDA_LAUNCH_BLOCKING"]="1"
import d3rlpy

dataset, env = d3rlpy.datasets.get_minari('atari/pong/expert-v0')

/gpfs/data/fs72297/jklotz/.conda/envs/d3rlpy_dev_requirements_py310/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]


2025-07-29 10:55.40 [info     ] Signatures have been automatically determined. action_signature=Signature(dtype=[dtype('int64')], shape=[(1,)]) observation_signature=Signature(dtype=[dtype('uint8')], shape=[(3, 210, 160)]) reward_signature=Signature(dtype=[dtype('float64')], shape=[(1,)])
2025-07-29 10:55.40 [info     ] Action-space has been automatically determined. action_space=<ActionSpace.DISCRETE: 2>
2025-07-29 10:55.40 [info     ] Action size has been automatically determined. action_size=6


In [3]:
env.observation_space

Box(0, 255, (210, 160, 3), uint8)

In [7]:
one,two = env.reset()

In [8]:
print(type(one))
print(type(two))

<class 'numpy.ndarray'>
<class 'dict'>


In [10]:
print((one.shape))
print((two.keys()))

(210, 160, 3)
dict_keys(['lives', 'episode_frame_number', 'frame_number'])


In [8]:
for ep in dataset.episodes:
    print(type(ep.observations))
    print(type(ep))
    break

<class 'numpy.ndarray'>
<class 'd3rlpy.dataset.components.Episode'>
